In [22]:
import random

In [23]:
RANKS = "23456789TJQKA"
SUITS = "♠♥♣♦"
SMALL_BLIND = 1
BIG_BLIND = 2


def newDeck():
    deck = [f"{rank}{suit}" for rank in RANKS for suit in SUITS]
    random.shuffle(deck)
    return deck

In [24]:
class Player:
    def __init__(self, name, chips):
        self.name = name
        self.chips = chips
        self.hand = []
        self.hasActedThisRound = False
        self.currentBet = 0
        self.isFolded = False
        self.isAllIn = False

    def resetHand(self):
        self.hand = []
        self.hasActedThisRound = False
        self.currentBet = 0
        self.isFolded = False
        self.isAllIn = False

In [ ]:
class PokerGame:
    def __init__(self, players):
        self.players = [Player(name, chips) for name, chips in players]

        self.smallBlind = SMALL_BLIND
        self.bigBlind = BIG_BLIND

        self.deck = []
        self.communityCards = []

        self.pot = 0
        self.currentPlayerPosition = 0

        self.gameState = "null"
        self.roundMaxBet = 0

    def currentPlayer(self):
        return self.players[self.currentPlayerPosition]

    def advancePlayer(self):
        self.currentPlayerPosition = (self.currentPlayerPosition + 1) % len(self.players)

    def activePlayers(self):
        return [player for player in self.players if not player.isFolded]

    def checkValidBet(self, player, bet):
        if bet == -1:
            player.isFolded = True
            return True

        if bet == player.chips:
            player.isAllIn = True
            return True
        
        if player.currentBet + bet < self.roundMaxBet:
            print(f"Invalid bet.")
            return False
        
        if bet > player.chips:
            print(f"Invalid bet, please put exact chip value for all-in.")
            return False
        
        return True

    def action(self, player):
        print(f"Current bet is ${self.roundMaxBet}")
        while True:
            try:
                bet = int(input(f"{player.name}, enter your bet, -1 to fold: "))
            except ValueError:
                continue

            if self.checkValidBet(player, bet):
                return bet

    def playerTurn(self):
        player = self.currentPlayer()

        if (player.isFolded or player.isAllIn):
            self.advancePlayer()
            return
        
        print(f"{player.name}'s turn")
        print(f"Player hand: {player.hand}")

        bet = self.action(player)

        # If folded this turn
        if bet == -1:
            player.hasActedThisRound = True
            self.advancePlayer()
            return
        
        # Apply their bet
        player.chips -= bet
        player.currentBet += bet
        self.pot += bet
        player.hasActedThisRound = True

        # If raised this turn
        if player.currentBet > self.roundMaxBet:
            self.roundMaxBet = player.currentBet
            for p in self.players:
                if ((not p.isFolded) and (not p.isAllIn)):
                    p.hasActedThisRound = False
            player.hasActedThisRound = True

        self.advancePlayer()

    def isBettingRoundComplete(self):
        if len(self.activePlayers()) <= 1:
            return True
        
        if not all(player.hasActedThisRound for player in self.activePlayers()):
            return False
        
        return all(((player.currentBet == self.roundMaxBet) or player.isAllIn) for player in self.activePlayers())

    def resetBettingRound(self):
        self.roundMaxBet = 0
        self.currentPlayerPosition = 0
        for player in self.players:
            player.currentBet = 0
            if ((not player.isAllIn) and (not player.isFolded)):
                player.hasActedThisRound = False

    def bettingRound(self):
        while not self.isBettingRoundComplete():
            self.playerTurn()

        print("Betting round complete.")

    def dealHoleCards(self):
        self.resetBettingRound()
        for _ in range(2):
            for player in self.players:
                player.hand.append(self.deck.pop(0))

    def postBlinds(self):
        sb = self.players[0]
        bb = self.players[1]

        sb.chips -= self.smallBlind
        sb.currentBet = self.smallBlind
        print(f"Player {sb.name} posted small blind of {self.smallBlind}.")
        self.advancePlayer()

        bb.chips -= self.bigBlind
        bb.currentBet = self.bigBlind
        print(f"Player {bb.name} posted big blind of {self.bigBlind}.")
        self.advancePlayer()

        self.pot += self.smallBlind + self.bigBlind
        self.roundMaxBet = self.bigBlind

    def dealFlop(self):
        self.resetBettingRound()
        self.deck.pop(0)  # Burn card
        for _ in range(3):
            self.communityCards.append(self.deck.pop(0))     

    def dealTurn(self):
        self.resetBettingRound()
        self.deck.pop(0)  # Burn card
        self.communityCards.append(self.deck.pop(0))

    def dealRiver(self):
        self.resetBettingRound()
        self.deck.pop(0)  # Burn card
        self.communityCards.append(self.deck.pop(0))    


    def startHand(self):
        self.deck = newDeck()
        self.communityCards = []
        self.pot = 0

        for player in self.players:
            player.resetHand()

        self.dealHoleCards()
        self.postBlinds()
        self.gameState = "Pre-flop"
        self.bettingRound()

        self.dealFlop()
        self.gameState = "Flop"
        self.bettingRound()

        self.dealTurn()
        self.gameState = "Turn"
        self.bettingRound()

        self.dealRiver()
        self.gameState = "River"
        self.bettingRound()

In [26]:
game = PokerGame([("Player1", 1000), ("Player2", 1000), ("Player3", 1000)])
game.startHand()

Player Player1 posted small blind of 1.
Player Player2 posted big blind of 2.
Player3's turn
Player hand: ['J♦', '5♠']
Current bet is $2
Player1's turn
Player hand: ['6♥', '8♦']
Current bet is $2
Player2's turn
Player hand: ['5♥', 'Q♠']
Current bet is $2
Betting round complete.
Player1's turn
Player hand: ['6♥', '8♦']
Current bet is $0
Invalid bet.
Player2's turn
Player hand: ['5♥', 'Q♠']
Current bet is $10000
Player3's turn
Player hand: ['J♦', '5♠']
Current bet is $10000
Betting round complete.
Player1's turn
Player hand: ['6♥', '8♦']
Current bet is $0


KeyboardInterrupt: Interrupted by user